In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import random
from collections import deque
import torch
import torch.nn as nn
import torch.optim as optim

random.seed(24)
np.random.seed(24)

### CUDA Setup

In [ ]:
device = torch.device("cuda")
print(f"Using device: {device}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

torch.manual_seed(24)
torch.cuda.manual_seed_all(24)

---
## Markov Decision Process (MDP) Framework

In [ ]:
class GridWorld:
    """
    A simple 4x4 GridWorld MDP.
    State: (row, col). Goal: reach (3,3). Pit: (2,2).
    Actions: 0=Up, 1=Down, 2=Left, 3=Right
    """
    def __init__(self, size=4):
        self.size = size
        self.n_states = size * size
        self.n_actions = 4
        self.goal = (size-1, size-1)
        self.pit = (size//2, size//2)
        self.action_effects = [(-1,0),(1,0),(0,-1),(0,1)]  # U D L R

    def state_id(self, s): return s[0]*self.size + s[1]
    def state_xy(self, sid): return (sid // self.size, sid % self.size)

    def reset(self):
        self.pos = (0, 0)
        return self.state_id(self.pos)

    def step(self, action):
        dr, dc = self.action_effects[action]
        r, c = self.pos
        nr = max(0, min(self.size-1, r + dr))
        nc = max(0, min(self.size-1, c + dc))
        self.pos = (nr, nc)
        done = False
        if self.pos == self.goal:
            reward, done = +10.0, True
        elif self.pos == self.pit:
            reward, done = -10.0, True
        else:
            reward = -0.1  # small step penalty encourages efficiency
        return self.state_id(self.pos), reward, done

env = GridWorld()
print(f"GridWorld: {env.n_states} states, {env.n_actions} actions")
print(f"Goal state id: {env.state_id(env.goal)}, Pit state id: {env.state_id(env.pit)}")

---
## Deep Q-Network (DQN)

DQN extends Q-learning to large/continuous state spaces using:
1. **Neural network** to approximate $Q(s,a;\theta)$
2. **Experience replay buffer** $\mathcal{B}$ to break temporal correlations
3. **Target network** $Q(\cdot;\theta^-)$ updated every $T_\text{target}$ steps

Loss:
$$L_\text{DQN}(\theta, \theta^-) = \frac{1}{B} \sum_{i=1}^B \bigl\| Q(s_i,a_i;\theta) - \tilde{Y}_i^Q \bigr\|_2^2,
\quad \tilde{Y}_i^Q = r_i + \gamma \max_{a'} Q(s'_i,a';\theta^-)$$

In [ ]:

class QNetwork(nn.Module):
    """Fully-connected Q-network: state -> Q-values for all actions."""
    def __init__(self, n_states, n_actions, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_states, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, n_actions)
        )

    def forward(self, x):
        if x.dim() == 1:
            x = x.unsqueeze(0)
        return self.net(x)


class ReplayBuffer:
    """Replay buffer that keeps encoded state tensors on the selected device."""
    def __init__(self, capacity=10000, device=device):
        self.buf = deque(maxlen=capacity)
        self.device = device

    def push(self, s, a, r, s_next, done):
        self.buf.append((
            s.detach().clone(),
            int(a),
            float(r),
            s_next.detach().clone(),
            float(done),
        ))

    def sample(self, batch_size):
        batch = random.sample(self.buf, batch_size)
        s, a, r, s_next, done = zip(*batch)
        return (
            torch.stack(s),
            torch.tensor(a, dtype=torch.long, device=self.device),
            torch.tensor(r, dtype=torch.float32, device=self.device),
            torch.stack(s_next),
            torch.tensor(done, dtype=torch.float32, device=self.device),
        )

    def __len__(self):
        return len(self.buf)


def dqn(env, num_episodes=1000, gamma=0.99, lr=1e-3, batch_size=64, buffer_capacity=10000,
        epsilon_start=1.0, epsilon_end=0.05, epsilon_decay=0.995, target_update_freq=20, device=device):
    """
    Deep Q-Network (DQN) with consistent device placement.
    Networks, replay samples, one-hot states, and optimization stay on the selected device.
    """
    n_s, n_a = env.n_states, env.n_actions

    q_net = QNetwork(n_s, n_a).to(device)
    tgt_net = QNetwork(n_s, n_a).to(device)
    tgt_net.load_state_dict(q_net.state_dict())
    tgt_net.eval()

    optimizer = optim.Adam(q_net.parameters(), lr=lr)
    replay = ReplayBuffer(buffer_capacity, device=device)
    loss_fn = nn.MSELoss()
    eye_states = torch.eye(n_s, dtype=torch.float32, device=device)

    def encode(s):
        return eye_states[s]

    epsilon = epsilon_start
    cum_rewards = []

    for ep in range(num_episodes):
        s = env.reset()
        done = False
        total_r = 0.0

        while not done:
            state_t = encode(s)

            if random.random() < epsilon:
                a = random.randrange(n_a)
            else:
                with torch.no_grad():
                    q_vals = q_net(state_t)
                a = int(q_vals.argmax(dim=1).item())

            s_next, r, done = env.step(a)
            next_state_t = encode(s_next)
            total_r += r
            replay.push(state_t, a, r, next_state_t, done)
            s = s_next

            if len(replay) >= batch_size:
                s_b, a_b, r_b, sn_b, d_b = replay.sample(batch_size)

                q_curr = q_net(s_b).gather(1, a_b.unsqueeze(1)).squeeze(1)

                with torch.no_grad():
                    q_next = tgt_net(sn_b).max(dim=1).values
                    q_tgt = r_b + gamma * q_next * (1.0 - d_b)

                loss = loss_fn(q_curr, q_tgt)
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                optimizer.step()

        if (ep + 1) % target_update_freq == 0:
            tgt_net.load_state_dict(q_net.state_dict())

        epsilon = max(epsilon_end, epsilon * epsilon_decay)
        cum_rewards.append(total_r)

    return q_net, cum_rewards


dqn_net, rewards_dqn = dqn(env, num_episodes=1000, device=device)
print(f"Q-network parameters live on: {next(dqn_net.parameters()).device}")

window = 50
reward_tensor = torch.tensor(rewards_dqn, dtype=torch.float32, device=device)
kernel = torch.ones(window, dtype=torch.float32, device=device) / window
smooth_dqn = torch.nn.functional.conv1d(
    reward_tensor.view(1, 1, -1),
    kernel.view(1, 1, -1)
).view(-1)
smooth_dqn_cpu = smooth_dqn.detach().cpu().numpy()

plt.figure(figsize=(10, 4))
plt.plot(smooth_dqn_cpu, color='crimson')
plt.xlabel('Episode')
plt.ylabel('Cumulative Reward (50-ep smoothed)')
plt.title(f'DQN: Deep Q-Network Training on GridWorld ({device.type.upper()})')
plt.tight_layout()
plt.show()
print(f"DQN final smoothed reward: {smooth_dqn[-1].item():.3f}")